# What Is a Tensor? — see it in `numpy`

A **tensor is just a multidimensional array of numbers**. This notebook makes that concrete:
you'll read a tensor's **rank** (number of dimensions) and **shape**, compute its **memory
footprint**, and watch why adding *one dimension* takes a tensor from megabytes to past a whole
chip's HBM.

Pure `numpy` + `matplotlib`. Runs on **CPU** — no GPU/TPU needed.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 1. Rank, shape, size — straight off the array

`ndim` *is* the rank: how many indices you need to reach one number. `shape` is the size of each
dimension; `size` is their product (the element count).


In [ ]:
scalar = np.array(3.14)                 # rank 0
vector = np.array([4, 3, 0])            # rank 1  (Fleisch's arrow)
matrix = np.arange(9).reshape(3, 3)     # rank 2
cube   = np.arange(27).reshape(3, 3, 3) # rank 3

for name, t in [('scalar', scalar), ('vector', vector), ('matrix', matrix), ('cube', cube)]:
    print(f'{name:7s} rank(ndim)={t.ndim}  shape={str(t.shape):12s} size={t.size:3d}  bytes={t.nbytes}')


Note `vector = [4, 3, 0]` is exactly the array the *vector-basis-explorer* sim builds from the
arrow's shadows — the chip stores this flat list, never an arrow. And rank here means **number of
dimensions**, not the linear-algebra 'matrix rank' (independent rows) — same word, unrelated idea.


## 2. The footprint formula

A tensor's memory is just `size * bytes_per_element`, and `size` is the **product** of the
dimensions. So each dimension you add *multiplies* the footprint — it doesn't add to it. We
compute this **analytically** (model-scale tensors are far too big to actually allocate).


In [ ]:
BF16 = 2  # bytes per element for bfloat16, the default training dtype

def footprint_bytes(shape, bytes_per=BF16):
    size = 1
    for d in shape:
        size *= d
    return size * bytes_per

def human(n):
    for unit in ['B','KiB','MiB','GiB','TiB']:
        if n < 1024 or unit == 'TiB':
            return f'{n:.0f} {unit}' if n >= 10 or unit=='B' else f'{n:.1f} {unit}'
        n /= 1024

DIM = 4096  # a realistic model width
for rank in range(4):
    shape = (DIM,) * rank
    b = footprint_bytes(shape)
    print(f'rank {rank}  shape={str(shape):28s} -> {human(b):>9s}')


There it is: rank 1 is **8 KiB**, rank 2 is **32 MiB**, rank 3 is **128 GiB**. One extra
dimension took the same kind of data past a single chip's HBM (~95 GB).


## 3. The blow-up, charted

Plotting footprint against rank on a **log** axis — each step up is a clean multiplicative jump,
and rank 3 punches through the HBM ceiling.


In [ ]:
HBM_GB = 95
ranks = list(range(4))
bytes_ = [footprint_bytes((DIM,)*r) for r in ranks]
gib = [b / 1024**3 for b in bytes_]

fig, ax = plt.subplots(figsize=(7, 4.2))
bars = ax.bar(ranks, gib, color=['#94a3b8','#60a5fa','#3b82f6','#dc2626'])
ax.set_yscale('log')
ax.axhline(HBM_GB, color='#dc2626', ls='--', lw=1.5)
ax.text(0, HBM_GB*1.3, f'one chip HBM ~{HBM_GB} GB', color='#dc2626', fontsize=9)
ax.set_xticks(ranks)
ax.set_xticklabels([f'rank {r}\n{(DIM,)*r}' for r in ranks], fontsize=8)
ax.set_ylabel('footprint (GiB, log scale)')
ax.set_title(f'Each dimension multiplies the memory  (dim={DIM}, bf16)')
for bar, b in zip(bars, bytes_):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(), human(b),
            ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()


## 4. A real tensor that bites: the activation

Real activation tensors carry **batch × sequence × hidden** — every one a multiplier. Watch what
doubling the sequence length does, and why a longer-context job can OOM when a shorter one fit.


In [ ]:
def activation(batch, seq, hidden):
    b = footprint_bytes((batch, seq, hidden))
    print(f'[batch={batch}, seq={seq}, hidden={hidden}] (rank 3) -> {human(b)}')
    return b

a = activation(8, 8192, 4096)
b = activation(8, 16384, 4096)   # double the context
print(f'\ndoubling sequence length: {human(a)} -> {human(b)}  ({b/a:.0f}x)')
print('...and that is just ONE activation tensor, per layer, before optimizer state.')


## Takeaways

- A **tensor is a multidimensional array**; `ndim` is its **rank**, `shape` the size of each dimension.
- Footprint = `size * bytes`, and `size` is the **product** of the dimensions — so each added
  dimension **multiplies** memory, it doesn't add.
- That multiplicative growth is why **rank**, not just size, drives the HBM bill — and why
  activation and KV-cache tensors devour memory on longer contexts and bigger batches.
- Next: *tensor shapes* — why those dimensions must be multiples of 128, and how padding inflates
  the footprint further.
